# SAQ Relative Error Plots

Plots SAQ project-schema CSVs from the latest SAQ runs.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 40,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 21,
})
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

PROJECT_ROOT = Path("/home/cpanourg/projects/2-hdvc")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/urania_results/results/saq")
FIGURES_DIR = DATA_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METHODS_TO_PLOT = ["SAQ"]
DATASETS_TO_PLOT = ["deep", "bigann", "gist", "msmarco", "openai"]
FIGURE_SAVE_FORMATS = ("pdf", "svg")

ADC_TIME_COL = "adc_time_per_pair_s"
ADC_TIME_LABEL = "ADC time per pair (s)"
CONFIG_COLS = ["method", "dataset", "nbits", "bits_per_vector"]


In [ ]:
COLOR_PALETTE = [
    "tab:blue", "tab:green", "tab:purple", "tab:orange", "tab:red",
    "tab:brown", "tab:pink", "tab:gray", "tab:olive", "tab:cyan",
]
MARKER_PALETTE = ["o", "v", "s", "^", "D", "<", ">", "p", "*", "h"]


def save_figure(fig, output_dir: Path, stem: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in FIGURE_SAVE_FORMATS:
        path = output_dir / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight")
        paths.append(path)
    print("Saved " + " and ".join(str(p) for p in paths))


def style_axes(ax, tick_fontsize=40, grid_axis="y"):
    ax.grid(True, axis=grid_axis, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=tick_fontsize)


def adjust_ylabel_position(ax, y_label: str):
    if y_label == "Avg Relative Error":
        ax.yaxis.set_label_coords(-0.13, 0.39)


def set_sci_axes(ax):
    for axis in [ax.xaxis, ax.yaxis]:
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((-2, 3))
        axis.set_major_formatter(fmt)


In [ ]:
def load_saq_data(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    csv_paths = sorted(data_dir.glob("*_SAQ_adc_vs_exact_eval.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No SAQ CSV files found in {data_dir}")

    frames = []
    for path in csv_paths:
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = path.name.split("_")[0]
        if "method" not in df.columns:
            df["method"] = "SAQ"
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    numeric_cols = [
        "bits_per_vector", "nbits", "train_size", "adc_time_s", "rel_error_mean", "rel_error_std",
        "train_time_s", "encoding_time_s", "distance_table_time_s", "cdist_time_s",
        "nb_sample", "nq_sample", "dim", "nb", "nq", "n_subquantizers",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["method"] = df["method"].replace({"SAQ": "SAQ"})
    df["pair_count"] = df["nb_sample"] * df["nq_sample"]
    df[ADC_TIME_COL] = df["adc_time_s"] / df["pair_count"]
    df["bpv_ratio"] = df["bits_per_vector"] / df["dim"]

    required = ["method", "dataset", "bits_per_vector", "nbits", "adc_time_s", "rel_error_mean"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    return df

raw_df = load_saq_data(DATA_DIR)
print("Rows per dataset:")
display(raw_df.groupby("dataset").size())
raw_df.sort_values(["dataset", "bits_per_vector"]).head(20)


In [ ]:
def aggregate_metric_df(df: pd.DataFrame, y_cols=("rel_error_mean", ADC_TIME_COL)) -> pd.DataFrame:
    agg_spec = {c: "mean" for c in y_cols if c in df.columns}
    for c in [
        "rel_error_std", "distance_table_time_s", "cdist_time_s",
        "train_time_s", "encoding_time_s", "dim", "nb", "nb_sample",
        "nq_sample", "pair_count", "adc_time_s", "bpv_ratio", "train_size",
    ]:
        if c in df.columns and c not in agg_spec:
            agg_spec[c] = "mean" if pd.api.types.is_numeric_dtype(df[c]) else "first"
    return (
        df.groupby(CONFIG_COLS, as_index=False)
        .agg(agg_spec)
        .sort_values(["dataset", "bits_per_vector"])
    )

plot_df = aggregate_metric_df(raw_df)
plot_df.groupby("dataset").size()


In [ ]:
def plot_metric_vs_bpv(
    df: pd.DataFrame,
    y_col: str,
    y_label: str,
    output_stem: str,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(
                subset=["bits_per_vector", y_col]
            ).copy()
            if sub.empty:
                print(f"Skipping {method}/{dataset}: no rows")
                continue
            sub = sub.sort_values("bits_per_vector")
            fig, ax = plt.subplots()
            yerr = sub["rel_error_std"] if y_col == "rel_error_mean" and "rel_error_std" in sub else None
            ax.errorbar(
                sub["bits_per_vector"], sub[y_col], yerr=yerr,
                fmt="o-", color=COLOR_PALETTE[0], markersize=16,
                linewidth=2.5, markeredgewidth=2, markeredgecolor="black",
                capsize=5 if yerr is not None else 0, capthick=2, elinewidth=1.5,
            )
            ax.set_xlabel("Bits per vector", fontsize=40)
            ax.set_ylabel(y_label, fontsize=40)
            adjust_ylabel_position(ax, y_label)
            vals = sorted(sub["bits_per_vector"].dropna().unique())
            ax.set_xticks(vals)
            ax.set_xticklabels([str(int(v)) for v in vals], rotation=0)
            style_axes(ax, tick_fontsize=34, grid_axis="y")
            set_sci_axes(ax)
            plt.tight_layout()
            stem = f"{output_stem}_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)

plot_metric_vs_bpv(plot_df, "rel_error_mean", "Avg Relative Error", "relerr_vs_bits_per_vector")
plot_metric_vs_bpv(plot_df, ADC_TIME_COL, ADC_TIME_LABEL, "adc_time_vs_bits_per_vector")
plot_metric_vs_bpv(plot_df, "encoding_time_s", "Encoding time (s)", "encoding_time_vs_bits_per_vector")


In [ ]:
def pareto_frontier_minimize(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    pts = df.sort_values([x_col, y_col]).copy()
    frontier_rows = []
    best_y = np.inf
    for _, row in pts.iterrows():
        if row[y_col] < best_y:
            frontier_rows.append(row)
            best_y = row[y_col]
    if not frontier_rows:
        return pts.iloc[0:0]
    return pd.DataFrame(frontier_rows)


def plot_pareto_adc_vs_relerr(
    df: pd.DataFrame,
    datasets=DATASETS_TO_PLOT,
    methods=METHODS_TO_PLOT,
    output_dir: Path = FIGURES_DIR,
):
    for method in methods:
        for dataset in datasets:
            sub = df[(df["method"] == method) & (df["dataset"] == dataset)].dropna(
                subset=[ADC_TIME_COL, "rel_error_mean", "bits_per_vector"]
            ).copy()
            if sub.empty:
                print(f"Skipping {method}/{dataset}: no rows")
                continue
            sub = sub.sort_values([ADC_TIME_COL, "rel_error_mean"])
            frontier = pareto_frontier_minimize(sub, ADC_TIME_COL, "rel_error_mean")

            fig, ax = plt.subplots()
            ax.scatter(
                sub[ADC_TIME_COL], sub["rel_error_mean"],
                color="tab:blue", marker="o", s=260,
                edgecolors="black", linewidths=2, alpha=0.75,
            )
            if len(frontier) > 0:
                ax.plot(frontier[ADC_TIME_COL], frontier["rel_error_mean"], "-", color="gray", linewidth=1.5, alpha=0.8)
                ax.scatter(
                    frontier[ADC_TIME_COL], frontier["rel_error_mean"],
                    color="#D32F2F", marker="o", s=330,
                    edgecolors="black", linewidths=2, zorder=3,
                )
            for _, row in frontier.iterrows():
                label = f"{int(row['bits_per_vector'])} bpv"
                ax.annotate(
                    label, (row[ADC_TIME_COL], row["rel_error_mean"]),
                    xytext=(10, 10), textcoords="offset points", fontsize=13,
                    bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="none", alpha=0.72),
                    arrowprops=dict(arrowstyle="-", color="0.35", lw=0.8, shrinkA=0, shrinkB=5),
                )
            ax.set_xlabel(ADC_TIME_LABEL, fontsize=40)
            ax.set_ylabel("Avg Relative Error", fontsize=40)
            adjust_ylabel_position(ax, "Avg Relative Error")
            style_axes(ax, tick_fontsize=34, grid_axis="both")
            set_sci_axes(ax)
            plt.tight_layout()
            stem = f"pareto_relerr_vs_adc_time_{method.lower()}_{dataset}"
            save_figure(fig, output_dir, stem)
            plt.show()
            plt.close(fig)

plot_pareto_adc_vs_relerr(plot_df)


## Dataset-Separated Plots Only

This notebook intentionally avoids figures that put multiple datasets on the same axes. Each plot below is generated per dataset; cross-dataset comparisons should be made by placing the saved figures side by side.


## Analytical Hyperparameter Views

The current SAQ export varies `nbits`; `bits_per_vector` is derived from `dim * nbits`, so datasets with different dimensionality occupy different memory budgets even at the same SAQ bit-width. The cells below make that distinction explicit and summarize every logged SAQ/configuration column that behaves like a hyperparameter or run setting.


In [ ]:
HYPERPARAM_COLS = [
    "nbits", "bits_per_vector", "bpv_ratio", "dim", "n_subquantizers",
    "train_size", "nb_sample", "nq_sample", "sample_mode", "seed",
]

available_hyperparam_cols = [c for c in HYPERPARAM_COLS if c in raw_df.columns]


def summarize_hyperparams(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset, sub in df.groupby("dataset", sort=True):
        for col in available_hyperparam_cols:
            vals = sub[col].dropna().unique()
            try:
                vals = sorted(vals)
            except TypeError:
                vals = sorted(map(str, vals))
            if len(vals) <= 10:
                value_repr = ", ".join(str(int(v)) if isinstance(v, (int, np.integer, float, np.floating)) and float(v).is_integer() else str(v) for v in vals)
            else:
                value_repr = f"{len(vals)} values: [{vals[0]}, ..., {vals[-1]}]"
            rows.append({
                "dataset": dataset,
                "column": col,
                "n_unique": len(vals),
                "values": value_repr,
            })
    return pd.DataFrame(rows)

hyperparam_summary = summarize_hyperparams(raw_df)
hyperparam_summary_path = FIGURES_DIR / "saq_hyperparameter_inventory.csv"
hyperparam_summary.to_csv(hyperparam_summary_path, index=False)
print(f"Saved {hyperparam_summary_path}")
display(hyperparam_summary)


In [ ]:
def plot_metric_vs_nbits_per_dataset(
    df: pd.DataFrame,
    y_col: str,
    y_label: str,
    output_stem: str,
    output_dir: Path = FIGURES_DIR,
    datasets=DATASETS_TO_PLOT,
):
    for dataset in datasets:
        sub = df[(df["dataset"] == dataset) & (df["method"] == "SAQ")].dropna(
            subset=["nbits", y_col]
        ).sort_values("nbits")
        if sub.empty:
            print(f"Skipping {dataset}: no rows")
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(
            sub["nbits"], sub[y_col],
            marker="o", linestyle="-", color="tab:blue", markersize=14,
            linewidth=2.5, markeredgewidth=1.5, markeredgecolor="black",
        )
        ax.set_xlabel("SAQ bits per dimension (`nbits`)", fontsize=34)
        ax.set_ylabel(y_label, fontsize=36)
        adjust_ylabel_position(ax, y_label)
        ax.set_xticks(sorted(sub["nbits"].dropna().unique()))
        style_axes(ax, tick_fontsize=30, grid_axis="y")
        set_sci_axes(ax)
        plt.tight_layout()
        stem = f"{output_stem}_saq_{dataset}"
        save_figure(fig, output_dir, stem)
        png = output_dir / f"{stem}.png"
        fig.savefig(png, dpi=180, bbox_inches="tight")
        print(f"Saved {png}")
        plt.show()
        plt.close(fig)

plot_metric_vs_nbits_per_dataset(plot_df, "rel_error_mean", "Avg Relative Error", "relerr_vs_nbits")
plot_metric_vs_nbits_per_dataset(plot_df, ADC_TIME_COL, ADC_TIME_LABEL, "adc_time_per_pair_vs_nbits")
plot_metric_vs_nbits_per_dataset(plot_df, "encoding_time_s", "Encoding time (s)", "encoding_time_vs_nbits")


In [ ]:
def save_per_dataset_metric_tables(
    df: pd.DataFrame,
    output_dir: Path = FIGURES_DIR,
    datasets=DATASETS_TO_PLOT,
):
    cols = [
        "dataset", "nbits", "bits_per_vector", "rel_error_mean", "rel_error_std",
        ADC_TIME_COL, "encoding_time_s", "train_time_s", "distance_table_time_s", "cdist_time_s",
    ]
    cols = [c for c in cols if c in df.columns]
    for dataset in datasets:
        sub = df[(df["dataset"] == dataset) & (df["method"] == "SAQ")][cols].sort_values("nbits")
        if sub.empty:
            print(f"Skipping table for {dataset}: no rows")
            continue
        path = output_dir / f"saq_metric_table_{dataset}.csv"
        sub.to_csv(path, index=False)
        print(f"Saved {path}")

save_per_dataset_metric_tables(plot_df)


In [ ]:
def add_marginal_columns(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset, sub in df.sort_values("nbits").groupby("dataset", sort=True):
        sub = sub.copy().sort_values("nbits")
        sub["prev_rel_error_mean"] = sub["rel_error_mean"].shift(1)
        sub["prev_bits_per_vector"] = sub["bits_per_vector"].shift(1)
        sub["prev_adc_time_per_pair_s"] = sub[ADC_TIME_COL].shift(1)
        sub["relerr_reduction"] = sub["prev_rel_error_mean"] - sub["rel_error_mean"]
        sub["relerr_reduction_pct"] = sub["relerr_reduction"] / sub["prev_rel_error_mean"]
        sub["extra_bits_per_vector"] = sub["bits_per_vector"] - sub["prev_bits_per_vector"]
        sub["relerr_reduction_per_extra_bit"] = sub["relerr_reduction"] / sub["extra_bits_per_vector"]
        sub["adc_time_per_pair_delta_s"] = sub[ADC_TIME_COL] - sub["prev_adc_time_per_pair_s"]
        sub["error_time_product"] = sub["rel_error_mean"] * sub[ADC_TIME_COL]
        rows.append(sub)
    return pd.concat(rows, ignore_index=True)

analysis_df = add_marginal_columns(plot_df)
analysis_path = FIGURES_DIR / "saq_marginal_hyperparameter_analysis.csv"
analysis_df.to_csv(analysis_path, index=False)
print(f"Saved {analysis_path}")
display(analysis_df[[
    "dataset", "nbits", "bits_per_vector", "rel_error_mean", ADC_TIME_COL,
    "relerr_reduction_pct", "relerr_reduction_per_extra_bit", "error_time_product",
]].sort_values(["dataset", "nbits"]))


In [ ]:
def plot_marginal_improvement_per_dataset(
    df: pd.DataFrame,
    output_dir: Path = FIGURES_DIR,
    datasets=DATASETS_TO_PLOT,
):
    for dataset in datasets:
        sub = df[(df["dataset"] == dataset) & (df["method"] == "SAQ")].dropna(
            subset=["nbits", "relerr_reduction_pct"]
        ).sort_values("nbits")
        if sub.empty:
            print(f"Skipping marginal improvement for {dataset}: no rows")
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(
            sub["nbits"], 100.0 * sub["relerr_reduction_pct"],
            marker="o", linestyle="-", color="tab:green", markersize=14,
            linewidth=2.5, markeredgewidth=1.5, markeredgecolor="black",
        )
        ax.set_xlabel("SAQ bits per dimension (`nbits`)", fontsize=34)
        ax.set_ylabel("Rel. error reduction (%)", fontsize=32)
        ax.set_xticks(sorted(df["nbits"].dropna().unique()))
        style_axes(ax, tick_fontsize=30, grid_axis="y")
        set_sci_axes(ax)
        plt.tight_layout()
        stem = f"marginal_relerr_reduction_vs_nbits_saq_{dataset}"
        save_figure(fig, output_dir, stem)
        png = output_dir / f"{stem}.png"
        fig.savefig(png, dpi=180, bbox_inches="tight")
        print(f"Saved {png}")
        plt.show()
        plt.close(fig)

plot_marginal_improvement_per_dataset(analysis_df)


In [ ]:
def plot_error_time_product_per_dataset(
    df: pd.DataFrame,
    output_dir: Path = FIGURES_DIR,
    datasets=DATASETS_TO_PLOT,
):
    for dataset in datasets:
        sub = df[(df["dataset"] == dataset) & (df["method"] == "SAQ")].dropna(
            subset=["nbits", "error_time_product"]
        ).sort_values("nbits")
        if sub.empty:
            print(f"Skipping error/time product for {dataset}: no rows")
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(
            sub["nbits"], sub["error_time_product"],
            marker="o", linestyle="-", color="tab:purple", markersize=14,
            linewidth=2.5, markeredgewidth=1.5, markeredgecolor="black",
        )
        ax.set_xlabel("SAQ bits per dimension (`nbits`)", fontsize=34)
        ax.set_ylabel("Error x ADC time per pair", fontsize=30)
        ax.set_xticks(sorted(df["nbits"].dropna().unique()))
        style_axes(ax, tick_fontsize=30, grid_axis="y")
        set_sci_axes(ax)
        plt.tight_layout()
        stem = f"error_time_product_vs_nbits_saq_{dataset}"
        save_figure(fig, output_dir, stem)
        png = output_dir / f"{stem}.png"
        fig.savefig(png, dpi=180, bbox_inches="tight")
        print(f"Saved {png}")
        plt.show()
        plt.close(fig)

plot_error_time_product_per_dataset(analysis_df)


In [ ]:
def plot_time_breakdown_by_nbits(
    df: pd.DataFrame,
    output_dir: Path = FIGURES_DIR,
    datasets=DATASETS_TO_PLOT,
):
    time_cols = [c for c in ["train_time_s", "encoding_time_s", "distance_table_time_s", "cdist_time_s", "adc_time_s"] if c in df.columns]
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]
    for dataset in datasets:
        sub = df[(df["dataset"] == dataset) & (df["method"] == "SAQ")].dropna(subset=["nbits"]).sort_values("nbits")
        if sub.empty:
            print(f"Skipping time breakdown for {dataset}: no rows")
            continue
        x = np.arange(len(sub))
        width = 0.14
        fig, ax = plt.subplots(figsize=(12, 6.5))
        for i, col in enumerate(time_cols):
            ax.bar(x + (i - (len(time_cols) - 1) / 2) * width, sub[col], width=width, label=col, color=colors[i % len(colors)])
        ax.set_xticks(x)
        ax.set_xticklabels([str(int(v)) if float(v).is_integer() else str(v) for v in sub["nbits"]], fontsize=24)
        ax.set_xlabel("SAQ bits per dimension (`nbits`)", fontsize=30)
        ax.set_ylabel("Wall time (s)", fontsize=34)
        ax.set_yscale("log")
        style_axes(ax, tick_fontsize=24, grid_axis="y")
        ax.legend(frameon=True, fontsize=13, loc="best")
        plt.tight_layout()
        stem = f"time_breakdown_vs_nbits_saq_{dataset}"
        save_figure(fig, output_dir, stem)
        png = output_dir / f"{stem}.png"
        fig.savefig(png, dpi=180, bbox_inches="tight")
        print(f"Saved {png}")
        plt.show()
        plt.close(fig)

plot_time_breakdown_by_nbits(plot_df)


In [ ]:
def plot_dimension_memory_mapping_per_dataset(
    df: pd.DataFrame,
    output_dir: Path = FIGURES_DIR,
    datasets=DATASETS_TO_PLOT,
):
    for dataset in datasets:
        sub = df[(df["dataset"] == dataset) & (df["method"] == "SAQ")].dropna(
            subset=["nbits", "bits_per_vector"]
        ).sort_values("nbits")
        if sub.empty:
            print(f"Skipping memory mapping for {dataset}: no rows")
            continue
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(
            sub["nbits"], sub["bits_per_vector"],
            marker="o", linestyle="-", color="tab:orange", markersize=14,
            linewidth=2.5, markeredgewidth=1.5, markeredgecolor="black",
        )
        ax.set_xlabel("SAQ bits per dimension (`nbits`)", fontsize=34)
        ax.set_ylabel("Bits per vector", fontsize=36)
        ax.set_xticks(sorted(sub["nbits"].dropna().unique()))
        style_axes(ax, tick_fontsize=30, grid_axis="y")
        set_sci_axes(ax)
        plt.tight_layout()
        stem = f"bits_per_vector_mapping_vs_nbits_saq_{dataset}"
        save_figure(fig, output_dir, stem)
        png = output_dir / f"{stem}.png"
        fig.savefig(png, dpi=180, bbox_inches="tight")
        print(f"Saved {png}")
        plt.show()
        plt.close(fig)

plot_dimension_memory_mapping_per_dataset(plot_df)


## Raw Hyperparameter Slices

These plots avoid averaging across hyperparameter choices. One column is used as the x-axis and another as the legend. If repeated runs exist for the exact same setting, the function draws all points with a small deterministic jitter instead of collapsing them into a mean/std band.


In [ ]:
def _format_hp_value(value):
    if pd.isna(value):
        return "NA"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return str(int(value)) if float(value).is_integer() else f"{value:.4g}"
    return str(value)


def _sorted_unique_values(series: pd.Series):
    vals = list(series.dropna().unique())
    try:
        return sorted(vals)
    except TypeError:
        return sorted(vals, key=str)


def raw_relerr_slice_table(
    df: pd.DataFrame,
    x_col: str,
    legend_col: str,
    facet_col: Optional[str] = None,
    output_dir: Path = FIGURES_DIR,
) -> pd.DataFrame:
    cols = [c for c in [facet_col, x_col, legend_col, "dataset", "nbits", "bits_per_vector", "dim", "rel_error_mean", "rel_error_std", ADC_TIME_COL] if c]
    cols = list(dict.fromkeys([c for c in cols if c in df.columns]))
    out = df[cols].sort_values([c for c in [facet_col, legend_col, x_col] if c in cols]).copy()
    stem = f"raw_relerr_slice_x_{x_col}_legend_{legend_col}"
    if facet_col:
        stem += f"_facet_{facet_col}"
    path = output_dir / f"{stem}.csv"
    out.to_csv(path, index=False)
    print(f"Saved {path}")
    return out


def plot_raw_relerr_by_x_legend(
    df: pd.DataFrame,
    x_col: str,
    legend_col: str,
    facet_col: Optional[str] = None,
    output_stem: Optional[str] = None,
    output_dir: Path = FIGURES_DIR,
    y_col: str = "rel_error_mean",
):
    needed = [x_col, legend_col, y_col]
    if facet_col:
        needed.append(facet_col)
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"Skipping {x_col} / {legend_col}: missing {missing}")
        return []

    raw_relerr_slice_table(df.dropna(subset=[x_col, legend_col, y_col]), x_col, legend_col, facet_col, output_dir)
    facets = [None] if facet_col is None else _sorted_unique_values(df[facet_col])
    saved = []
    rng = np.random.default_rng(7)

    for facet in facets:
        sub = df.dropna(subset=[x_col, legend_col, y_col]).copy()
        if facet_col is not None:
            sub = sub[sub[facet_col] == facet]
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(11, 7))
        legend_values = _sorted_unique_values(sub[legend_col])
        numeric_x = pd.api.types.is_numeric_dtype(sub[x_col])
        x_span = float(sub[x_col].max() - sub[x_col].min()) if numeric_x and len(sub[x_col].unique()) > 1 else 1.0

        for idx, legend_value in enumerate(legend_values):
            g = sub[sub[legend_col] == legend_value].sort_values(x_col).copy()
            color = COLOR_PALETTE[idx % len(COLOR_PALETTE)]
            marker = MARKER_PALETTE[idx % len(MARKER_PALETTE)]
            repeated = g.duplicated(subset=[x_col], keep=False).any()
            if repeated and numeric_x:
                jitter = rng.uniform(-0.012, 0.012, size=len(g)) * x_span
                x_vals = g[x_col].astype(float).to_numpy() + jitter
                linestyle = "None"
            else:
                x_vals = g[x_col]
                linestyle = "-" if len(g) > 1 else "None"
            label = f"{legend_col}={_format_hp_value(legend_value)}"
            ax.plot(
                x_vals, g[y_col], linestyle=linestyle, marker=marker, color=color,
                markersize=12, linewidth=2.5, markeredgewidth=1.5,
                markeredgecolor="black", alpha=0.9, label=label,
            )

        ax.set_xlabel(x_col, fontsize=34)
        ax.set_ylabel("Avg Relative Error", fontsize=40)
        adjust_ylabel_position(ax, "Avg Relative Error")
        if numeric_x:
            vals = _sorted_unique_values(sub[x_col])
            if len(vals) <= 12:
                ax.set_xticks(vals)
                ax.set_xticklabels([_format_hp_value(v) for v in vals], rotation=0)
        else:
            ax.tick_params(axis="x", rotation=20)
        title_suffix = f" ({facet_col}={_format_hp_value(facet)})" if facet_col else ""
        ax.set_title(f"SAQ raw slices{title_suffix}", fontsize=28)
        style_axes(ax, tick_fontsize=26, grid_axis="y")
        set_sci_axes(ax)
        ax.legend(frameon=True, fontsize=13, loc="best")
        plt.tight_layout()

        stem = output_stem or f"raw_relerr_x_{x_col}_legend_{legend_col}"
        if facet_col is not None:
            stem = f"{stem}_{facet_col}_{_format_hp_value(facet)}".replace("/", "-").replace(" ", "_")
        save_figure(fig, output_dir, stem)
        png = output_dir / f"{stem}.png"
        fig.savefig(png, dpi=180, bbox_inches="tight")
        print(f"Saved {png}")
        saved.append(png)
        plt.show()
        plt.close(fig)
    return saved


In [ ]:
# Per-dataset raw views in the requested x-axis/legend form.
# Current SAQ has nbits as the only varied knob; bits_per_vector is dim * nbits.
plot_raw_relerr_by_x_legend(
    raw_df,
    x_col="bits_per_vector",
    legend_col="nbits",
    facet_col="dataset",
    output_stem="raw_relerr_vs_bits_per_vector_legend_nbits_saq",
)

plot_raw_relerr_by_x_legend(
    raw_df,
    x_col="nbits",
    legend_col="bits_per_vector",
    facet_col="dataset",
    output_stem="raw_relerr_vs_nbits_legend_bits_per_vector_saq",
)


In [ ]:
# Future-proof pairwise diagnostics for any additional SAQ knobs that appear in newer CSVs.
# This stays quiet for constants such as train_size/sample_mode in the current grid.
CANDIDATE_HP_COLS = [
    "nbits", "bits_per_vector", "min_bits", "max_bits", "n_segments", "segment_id",
    "quota", "space_quota", "train_size", "sample_mode", "seed",
]
varied_hp_cols = [
    c for c in CANDIDATE_HP_COLS
    if c in raw_df.columns and raw_df[c].dropna().nunique() > 1 and raw_df[c].dropna().nunique() <= 20
]
print("Varied hyperparameter-like columns:", varied_hp_cols)

for x_col in varied_hp_cols:
    for legend_col in varied_hp_cols:
        if x_col == legend_col:
            continue
        # Avoid duplicating the explicit plots above.
        if (x_col, legend_col) in {("nbits", "dataset"), ("bits_per_vector", "nbits")}: 
            continue
        if x_col in {"nbits", "bits_per_vector"} and legend_col in {"nbits", "bits_per_vector"}:
            continue
        plot_raw_relerr_by_x_legend(
            raw_df,
            x_col=x_col,
            legend_col=legend_col,
            facet_col="dataset" if "dataset" in raw_df.columns else None,
            output_stem=f"raw_relerr_vs_{x_col}_legend_{legend_col}_saq",
        )
